# Avaliação offline do reranker BGE

Use uma GPU L4. Este notebook não usa OpenAI nem Qdrant: ele avalia somente a reordenação do conjunto fixo de candidatos densos. Faça upload de `reranker_benchmark_v1.jsonl` quando solicitado.

In [ ]:
%pip -q install sentence-transformers huggingface-hub


In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded))
print(input_path)


In [ ]:
import json, math, random, time
import torch
from huggingface_hub import model_info
from sentence_transformers import CrossEncoder

MODEL_ID = 'BAAI/bge-reranker-v2-m3'
MODEL_REVISION = model_info(MODEL_ID).sha
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'device': DEVICE, 'model_revision': MODEL_REVISION})
model = CrossEncoder(MODEL_ID, revision=MODEL_REVISION, device=DEVICE, model_kwargs={'torch_dtype': 'auto'})
rows = [json.loads(line) for line in open(input_path, encoding='utf-8') if line.strip()]


In [ ]:
def rr(urls, expected):
    return next((1 / rank for rank, url in enumerate(urls, 1) if url == expected), 0.0)

before, after, cases = [], [], []
started = time.perf_counter()
for row in (r for r in rows if r['expected_status'] == 'matched'):
    candidates = row['candidates']
    baseline_urls = [c['url'] for c in candidates]
    pairs = [(row['query'], f"{c['title']}. {c['text']}") for c in candidates]
    scores = model.predict(pairs, batch_size=16, show_progress_bar=False)
    reranked = [c for c, _ in sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)]
    reranked_urls = [c['url'] for c in reranked]
    before.append(rr(baseline_urls, row['expected_url']))
    after.append(rr(reranked_urls, row['expected_url']))
    cases.append({'id': row['id'], 'baseline_rr': before[-1], 'reranked_rr': after[-1]})
elapsed = time.perf_counter() - started

rng = random.Random(42)
deltas = []
for _ in range(10_000):
    idx = [rng.randrange(len(before)) for _ in before]
    deltas.append(sum(after[i] - before[i] for i in idx) / len(idx))
deltas.sort()
improved = sum(a > b for a, b in zip(after, before))
worsened = sum(a < b for a, b in zip(after, before))
n_sign = improved + worsened
tail = sum(math.comb(n_sign, k) for k in range(min(improved, worsened) + 1)) / 2**n_sign if n_sign else .5
report = {
    'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'device': DEVICE, 'n': len(before),
    'baseline_mrr': sum(before) / len(before), 'reranked_mrr': sum(after) / len(after),
    'delta_mrr': {'estimate': sum(a-b for a,b in zip(after,before))/len(before), 'low': deltas[250], 'high': deltas[9750]},
    'improved': improved, 'worsened': worsened, 'unchanged': len(before)-improved-worsened,
    'exact_sign_test_p': min(1.0, 2*tail), 'elapsed_seconds': elapsed, 'cases': cases,
}
print({k:v for k,v in report.items() if k != 'cases'})
open('reranker_bge_v2_m3.json', 'w', encoding='utf-8').write(json.dumps(report, ensure_ascii=False, indent=2))
files.download('reranker_bge_v2_m3.json')
